In [1]:
import pandas as pd
import numpy as np
import pycountry
import pickle
from docx import Document
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.shared import Pt
import re
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
import math
import jenkspy
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
from scipy.interpolate import make_interp_spline
import sys,os

In [2]:
notebook_dir = os.path.dirname(os.getcwd())
source_data_path=os.path.join(notebook_dir, "Common Source Data")
sys.path.append(source_data_path)
from country_codes import countries

In [3]:
df=pd.read_csv('Supplementary Spreadsheet- Vaccination Coverage Estimates.csv')
df_incidence= pd.read_csv(os.path.join(notebook_dir, 'Disease Incidence','Supplementary Spreadsheet- Incidence Estimates.csv'))
df["Vaccination Coverage (%)"] = df["Vaccination Coverage (%)"].clip(upper=100)
df["Vaccination Coverage (%) Upper"] = df["Vaccination Coverage (%) Upper"].clip(upper=100)

#Filtering out Newcastle disease, to display only vND to avoid redundancy (WAHIS only reports vND, ND is duplicate to account for USA ND (non-vND) vaccination coverage)
df=df[df['Disease']!='Newcastle disease']


In [4]:
df=df.merge(df_incidence.drop(columns=['Country']),on=['ISO3','Year','Animal','Disease'],how='outer')

In [5]:
def find_vax_incidence_outliers(
    df: pd.DataFrame,
    z_thresh: float = 2.5,     # robust z-score cutoff
    use_point_estimates: bool = True
) -> pd.DataFrame:
    VAX   = 'Vaccination Coverage (%)'
    INC   = 'Incidence (Cases per 100,000)'
    SX    = 'Source_x'
    SY    = 'Source_y'

    #Filter rows for genuine reported data
    not_imputed = ~df[SX].fillna('').str.contains('Imputed', case=False) & \
                  ~df[SY].fillna('').str.contains('Imputed', case=False)

    not_all_vax_zero = ~(
        df[VAX].fillna(0).eq(0) 
    )

    not_all_inc_zero = ~(
        df[INC].fillna(0).eq(0) 
    )

    filt = df.loc[not_imputed & not_all_vax_zero & not_all_inc_zero].copy()

    vax_vals = filt[VAX].astype(float)
    inc_vals = filt[INC].astype(float)

    # Robust z-scores using median & MAD
    def robust_z(x):
        med = np.nanmedian(x)
        mad = np.nanmedian(np.abs(x - med))
        if mad == 0 or np.isnan(mad):
            sd = np.nanstd(x)
            if sd == 0 or np.isnan(sd):
                return pd.Series(np.zeros(len(x)), index=x.index)
            return (x - med) / sd
        return 0.6745 * (x - med) / mad

    z_vax = robust_z(vax_vals)
    z_inc = robust_z(inc_vals)


    outlier_mad = (z_vax >= z_thresh) & (z_inc >= z_thresh)

    pct_vax = vax_vals.rank(pct=True)
    pct_inc = inc_vals.rank(pct=True)
    joint_score = pct_vax.mul(pct_inc)  # larger = jointly more extreme

    res = filt.copy()
    res['Outlier rank'] = joint_score
    res['is_outlier_high_vax_and_inc'] = outlier_mad

    #Sort most “extreme” first
    res = res.sort_values(['is_outlier_high_vax_and_inc', 'is_outlier_high_vax_and_inc'],
                          ascending=[False, False])

    return res


In [6]:
out = find_vax_incidence_outliers(df, z_thresh=2.5)
outliers_only = out.loc[out['is_outlier_high_vax_and_inc']]

In [7]:
outliers_only=outliers_only[outliers_only['Year']==2025].sort_values(['Outlier rank'],ascending=False)
outliers_only["Outlier rank"] = outliers_only["Outlier rank"].rank(method="min", ascending=False).astype(int)


In [8]:
outliers_only.drop(columns=['is_outlier_high_vax_and_inc'],inplace=True)
outliers_only.rename(columns={'Source_x':'Source Vaccination Coverage',
                             'Source_y':'Source Disease Incidence',
                             'Year Data_x':'Year Vaccination Data',
                             'Year Data_y':'Year Incidence Data'},inplace=True)

In [9]:
#Removing instances in which collective cateogry "Influenza A virus (Inf. with)" is reported as well, but was reconciled only with HPAI
    #(Redundant in this case, as Influenza A virus category is set to equal cases in all other WAHIS influenza categories, and HPAI
        #is the only one reported)
IAV  = 'Influenza A virus (Inf. with)'
HPAI = 'High pathogenicity avian influenza viruses (poultry) (Inf. with)'
INC  = 'Incidence (Cases per 100,000)'

tmp = outliers_only.copy()

# Sets of equal incidence reports between HPAI and Influenza A collective category
hpai_keys = set(map(tuple,
    tmp.loc[tmp['Disease'].eq(HPAI), ['ISO3', INC]]
      .dropna()
      .to_numpy()
))

# Drop the redundant Influenza A virus category in this case
mask_drop_iav = tmp['Disease'].eq(IAV) & tmp[['ISO3', INC]].apply(tuple, axis=1).isin(hpai_keys)

outliers_only = tmp.loc[~mask_drop_iav].copy()

In [10]:
outliers_only.to_csv("Supplementary- High Vaccination, High Incidence Discordance.csv",index=False)